# Project 03: Movie Recommendation System

**Content-Based Filtering using CountVectorizer and Cosine Similarity**

**Objective:** Recommend the top 5 movies similar to a given title from the IMDB Top 1000 dataset, by vectorizing combined movie features (Genre, Director, and Stars) and measuring similarity between all 1000 movies — without using KNN.

**Author:** Melika Khaleidi Mofard

In [1]:
import pandas as pd
import numpy as np 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Step 1 — Load the Dataset

We load `imdb_top_1000.csv` and keep only the text features that describe each movie: `Series_Title`, `Genre`, `Director`, `Star1`, `Star2`, and `Star3`.


In [2]:
df = pd.read_csv('imdb_top_1000.csv')
df.head(5)

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


## Step 2 — Preprocess the Text Featuresm

Multi-word names like *Christopher Nolan* must stay as a **single token**, so spaces are replaced with underscores (`christopher_nolan`). This prevents the vectorizer from treating the first name and last name as separate, unrelated words. The `Genre` column contains commas (e.g. `Action, Crime, Drama`), which are also removed.

In [3]:
df.shape

(1000, 16)

In [4]:
df.columns.tolist()

['Poster_Link',
 'Series_Title',
 'Released_Year',
 'Certificate',
 'Runtime',
 'Genre',
 'IMDB_Rating',
 'Overview',
 'Meta_score',
 'Director',
 'Star1',
 'Star2',
 'Star3',
 'Star4',
 'No_of_Votes',
 'Gross']

In [5]:
need_columns=["Series_Title","Genre","IMDB_Rating","Director","Star1","Star2","Star3"]

In [6]:
df[need_columns].isna().sum()

Series_Title    0
Genre           0
IMDB_Rating     0
Director        0
Star1           0
Star2           0
Star3           0
dtype: int64

In [7]:
for i in need_columns[3:] :
    df[i] = df[i].astype(str).str.replace(' ','_')
df['Genre'] = df['Genre'].str.split(',')

In [8]:
df

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,[Drama],9.3,Two imprisoned men bond over a number of years...,80.0,Frank_Darabont,Tim_Robbins,Morgan_Freeman,Bob_Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"[Crime, Drama]",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis_Ford_Coppola,Marlon_Brando,Al_Pacino,James_Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"[Action, Crime, Drama]",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher_Nolan,Christian_Bale,Heath_Ledger,Aaron_Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"[Crime, Drama]",9.0,The early life and career of Vito Corleone in ...,90.0,Francis_Ford_Coppola,Al_Pacino,Robert_De_Niro,Robert_Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"[Crime, Drama]",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney_Lumet,Henry_Fonda,Lee_J._Cobb,Martin_Balsam,John Fiedler,689845,"4,360,000"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,https://m.media-amazon.com/images/M/MV5BNGEwMT...,Breakfast at Tiffany's,1961,A,115 min,"[Comedy, Drama, Romance]",7.6,A young New York socialite becomes interested ...,76.0,Blake_Edwards,Audrey_Hepburn,George_Peppard,Patricia_Neal,Buddy Ebsen,166544,NaN
996,https://m.media-amazon.com/images/M/MV5BODk3Yj...,Giant,1956,G,201 min,"[Drama, Western]",7.6,Sprawling epic covering the life of a Texas ca...,84.0,George_Stevens,Elizabeth_Taylor,Rock_Hudson,James_Dean,Carroll Baker,34075,NaN
997,https://m.media-amazon.com/images/M/MV5BM2U3Yz...,From Here to Eternity,1953,Passed,118 min,"[Drama, Romance, War]",7.6,"In Hawaii in 1941, a private is cruelly punish...",85.0,Fred_Zinnemann,Burt_Lancaster,Montgomery_Clift,Deborah_Kerr,Donna Reed,43374,"30,500,000"
998,https://m.media-amazon.com/images/M/MV5BZTBmMj...,Lifeboat,1944,NaN,97 min,"[Drama, War]",7.6,Several survivors of a torpedoed merchant ship...,78.0,Alfred_Hitchcock,Tallulah_Bankhead,John_Hodiak,Walter_Slezak,William Bendix,26471,NaN


In [9]:
str_need_columns = ["Genre","Director","Star1","Star2","Star3"]
df[str_need_columns]=df[str_need_columns].fillna('').astype(str)
df['Movie_vector']=df[str_need_columns].agg(''.join, axis=1)
df

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross,Movie_vector
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,['Drama'],9.3,Two imprisoned men bond over a number of years...,80.0,Frank_Darabont,Tim_Robbins,Morgan_Freeman,Bob_Gunton,William Sadler,2343110,"28,341,469",['Drama']Frank_DarabontTim_RobbinsMorgan_Freem...
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"['Crime', ' Drama']",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis_Ford_Coppola,Marlon_Brando,Al_Pacino,James_Caan,Diane Keaton,1620367,"134,966,411","['Crime', ' Drama']Francis_Ford_CoppolaMarlon_..."
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"['Action', ' Crime', ' Drama']",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher_Nolan,Christian_Bale,Heath_Ledger,Aaron_Eckhart,Michael Caine,2303232,"534,858,444","['Action', ' Crime', ' Drama']Christopher_Nola..."
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"['Crime', ' Drama']",9.0,The early life and career of Vito Corleone in ...,90.0,Francis_Ford_Coppola,Al_Pacino,Robert_De_Niro,Robert_Duvall,Diane Keaton,1129952,"57,300,000","['Crime', ' Drama']Francis_Ford_CoppolaAl_Paci..."
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"['Crime', ' Drama']",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney_Lumet,Henry_Fonda,Lee_J._Cobb,Martin_Balsam,John Fiedler,689845,"4,360,000","['Crime', ' Drama']Sidney_LumetHenry_FondaLee_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,https://m.media-amazon.com/images/M/MV5BNGEwMT...,Breakfast at Tiffany's,1961,A,115 min,"['Comedy', ' Drama', ' Romance']",7.6,A young New York socialite becomes interested ...,76.0,Blake_Edwards,Audrey_Hepburn,George_Peppard,Patricia_Neal,Buddy Ebsen,166544,NaN,"['Comedy', ' Drama', ' Romance']Blake_EdwardsA..."
996,https://m.media-amazon.com/images/M/MV5BODk3Yj...,Giant,1956,G,201 min,"['Drama', ' Western']",7.6,Sprawling epic covering the life of a Texas ca...,84.0,George_Stevens,Elizabeth_Taylor,Rock_Hudson,James_Dean,Carroll Baker,34075,NaN,"['Drama', ' Western']George_StevensElizabeth_T..."
997,https://m.media-amazon.com/images/M/MV5BM2U3Yz...,From Here to Eternity,1953,Passed,118 min,"['Drama', ' Romance', ' War']",7.6,"In Hawaii in 1941, a private is cruelly punish...",85.0,Fred_Zinnemann,Burt_Lancaster,Montgomery_Clift,Deborah_Kerr,Donna Reed,43374,"30,500,000","['Drama', ' Romance', ' War']Fred_ZinnemannBur..."
998,https://m.media-amazon.com/images/M/MV5BZTBmMj...,Lifeboat,1944,NaN,97 min,"['Drama', ' War']",7.6,Several survivors of a torpedoed merchant ship...,78.0,Alfred_Hitchcock,Tallulah_Bankhead,John_Hodiak,Walter_Slezak,William Bendix,26471,NaN,"['Drama', ' War']Alfred_HitchcockTallulah_Bank..."


## Step 3 — Vectorization and Similarity Matrix

`CountVectorizer` converts each movie's text vector into a numerical bag-of-words vector. Then `cosine_similarity` measures how similar every movie is to every other movie, producing a **1000 × 1000 similarity matrix**. Each value is a score between 0 and 1, where 1 means identical feature sets. Unlike KNN, no model is trained — similarity is computed directly between the vectors.


In [10]:
vectorizer = CountVectorizer()
model_train = vectorizer.fit_transform(df['Movie_vector'])
model = cosine_similarity(model_train)
model[:5,:5]

array([[1.        , 0.40824829, 0.35355339, 0.40824829, 0.35355339],
       [0.40824829, 1.        , 0.57735027, 0.66666667, 0.57735027],
       [0.35355339, 0.57735027, 1.        , 0.57735027, 0.5       ],
       [0.40824829, 0.66666667, 0.57735027, 1.        , 0.57735027],
       [0.35355339, 0.57735027, 0.5       , 0.57735027, 1.        ]])

## Step 4 — Recommendation Function

The function takes a movie title as input:

1. Finds the movie in the dataset (case-insensitive).
2. Reads its row from the similarity matrix.
3. Sorts all movies by their similarity score.
4. Removes the input movie itself (its score with itself is always 1).
5. Prints the top 5 most similar movies with their scores.


In [11]:
def recommentaion_5_movies(movie_name, num_recommendation=5):
    movie_name = movie_name.strip().lower()
    title = df['Series_Title'].str.lower()
    find_movie = df[movie_name == title]
    if find_movie.empty:
        print(f'We Cannot found <{movie_name}> !')
        return
    get_index_movie = find_movie.index[0]
    similatry = model[get_index_movie]
    sort_similatry = np.argsort(similatry)[::-1]
    most_5_similatry = [i for i in sort_similatry if i != get_index_movie][:num_recommendation]
    recommedation_movie = df.iloc[most_5_similatry]['Series_Title']
    for i, movie in enumerate(recommedation_movie):
        print(f'({i+1}){movie}')

## Step 5 — Take the Movie Title from the User

The program asks the user for a movie title and returns the 5 most similar movies based on the computed similarity matrix.


In [12]:
print('------------------Welcome to Recommender System---------------------')
movie_name =input('Enter the name of movie:')
print('\n')
recommentaion_5_movies(movie_name)
print('--------------------------------------------------------------------')

------------------Welcome to Recommender System---------------------


Enter the name of movie: Lifeboat




(1)Platoon
(2)Ivanovo detstvo
(3)Saving Private Ryan
(4)Judgment at Nuremberg
(5)The Deer Hunter
--------------------------------------------------------------------


## Results and Conclusion

The content-based recommendation system was built using the `Genre`, `Director`, `Star1`, `Star2`, and `Star3` features from the IMDB Top 1000 dataset.

After preprocessing the text data, `CountVectorizer` converted each movie's combined features into a numerical vector. Cosine similarity was then used to compare the movies and rank them by similarity.

When a user enters a movie title that exists in the dataset, the program returns the **five most similar movies**, along with their similarity scores. The input movie itself is excluded from the recommendations. If the title is not found, the program displays a message to the user.

This project uses **content-based filtering** and cosine similarity; it does not use KNN. The recommendations are based on shared genres, directors, and cast members, so they reflect similarity in the selected movie features.
